In [ ]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 1a. Generación de gráficos relacionados con ocupación
# DESCRIPCIÓN:
#   Este código carga datos del sensor del aula 1.6 (temperatura,
#   humedad, CO2) a partir de archivos Excel semanales. Calcula los
#   cambios temporales (deltas) de cada variable y genera:
#     - Gráficos de dispersión delta vs ocupación
#     - Gráficos de dispersión valor absoluto vs ocupación
#     - Gráfico de dispersión CO2 vs Humedad
#     - Boxplots por nivel de ocupación (Vacío, Baja, Media, Alta)
#   Separados por meses y por ubicación del sensor (pasillo).
# ===================================================================

import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import numpy as np

# Directorio de salida para los gráficos
OUTPUT_DIR = 'plots_scatters_deltas'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# CARGAR TODOS LOS DATOS
# ==========================================
file_pattern = '1.6_data_*_week_*.xlsx'
all_files = sorted(glob.glob(file_pattern))

if not all_files:
    print(f"\nERROR: No files found matching '{file_pattern}'")
    exit()

print(f"\nFound {len(all_files)} weekly files")

metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']

SENSORS   = ['Corridor']
VARIABLES = ['Temperature', 'CO2', 'Humidity']

# Color asignado al sensor del pasillo
sensor_colors = {
    'Corridor': '#2196F3',
}

month_order = ['February', 'March']

# Niveles de ocupación: límites y etiquetas
OCC_BINS   = [-1, 0, 10, 20, np.inf]
OCC_LABELS = ['Vacío (0)', 'Baja (1–10)', 'Media (11–20)', 'Alta (>20)']
OCC_COLORS = ['#90CAF9', '#42A5F5', '#1565C0', '#0D2A6E']

print("\nLOADING ALL DATA...")

all_records = []

for file_path in all_files:
    fname = os.path.basename(file_path)
    print(f"   {fname}...", end=" ")

    df = pd.read_excel(file_path)
    date_cols  = [c for c in df.columns if c not in metadata_cols]
    timestamps = pd.to_datetime(date_cols, errors='coerce')

    def get_series(variable, location):
        """Extrae la serie temporal de una variable y ubicación concretas."""
        mask = (df['Variable_Measure'] == variable) & (df['Sensor_Location'] == location)
        row  = df[mask]
        if len(row) > 0:
            return row[date_cols].iloc[0].astype(float).values
        return np.full(len(date_cols), np.nan)

    # Extraer ocupación
    occ_mask  = df['Variable_Measure'] == 'Occupancy'
    occ_row   = df[occ_mask]
    occupancy = occ_row[date_cols].iloc[0].astype(float).values if len(occ_row) > 0 else np.full(len(date_cols), np.nan)

    months = [t.month_name() if not pd.isnull(t) else None for t in timestamps]

    for sensor in SENSORS:
        for var in VARIABLES:
            values = get_series(var, sensor)
            for i, ts in enumerate(timestamps):
                if pd.isnull(ts):
                    continue
                all_records.append({
                    'timestamp': ts,
                    'month':     months[i],
                    'sensor':    sensor,
                    'variable':  var,
                    'value':     values[i],
                    'occupancy': occupancy[i],
                })

    print("OK")

df_all = pd.DataFrame(all_records)
df_all = df_all.sort_values('timestamp')

print(f"\n   Total records: {len(df_all)}")

# ==========================================
# CALCULAR DELTAS
# ==========================================
print("\nCalculating deltas...")

# Delta: diferencia entre el valor actual y el anterior para cada sensor+variable
df_all['delta_value'] = df_all.groupby(['sensor', 'variable'])['value'].diff()

# Asignar nivel de ocupación según los rangos definidos
df_all['occ_level'] = pd.cut(
    df_all['occupancy'],
    bins=OCC_BINS,
    labels=OCC_LABELS,
    right=True
)

unique_months = [m for m in month_order if m in df_all['month'].unique()]
df_all = df_all[df_all['month'].isin(unique_months)]

print(f"   Records after filter (Feb+Mar): {len(df_all)}")

# ==========================================
# FUNCIÓN: ΔVariable vs Ocupación
# ==========================================
def scatter_delta(y_var, y_label, suptitle, filename):
    """Genera un scatter de delta (incremento) de una variable frente a
    la ocupación instantánea, con una línea de tendencia por sensor."""
    month_pos = {m: i for i, m in enumerate(unique_months)}
    n_months  = len(unique_months)

    if n_months == 1:
        fig, axes = plt.subplots(1, 1, figsize=(7, 5))
        axes_flat = [axes]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes_flat = axes.flatten()

    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)

    any_data = False

    for month, pos in month_pos.items():
        ax = axes_flat[pos]
        ax.set_title(month, fontsize=12, fontweight='bold')
        ax.set_xlabel('Ocupación (personas)', fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.5)
        ax.axvline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.5)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)

        handles = []

        for sensor in SENSORS:
            sub = df_all[
                (df_all['month'] == month) &
                (df_all['sensor'] == sensor) &
                (df_all['variable'] == y_var)
            ].dropna(subset=['occupancy', 'delta_value'])

            if len(sub) == 0:
                continue

            any_data = True
            color = sensor_colors.get(sensor, '#999999')

            sc = ax.scatter(sub['occupancy'], sub['delta_value'],
                            c=color, alpha=0.45, s=10, edgecolors='none',
                            label=sensor)
            handles.append(sc)

            # Línea de tendencia lineal
            try:
                x_v = sub['occupancy'].values
                y_v = sub['delta_value'].values
                if len(x_v) > 1:
                    z   = np.polyfit(x_v, y_v, 1)
                    p   = np.poly1d(z)
                    x_t = np.linspace(x_v.min(), x_v.max(), 100)
                    ax.plot(x_t, p(x_t), color=color, linewidth=1.6,
                            linestyle='--', alpha=0.9)
            except Exception:
                pass

        if handles:
            ax.legend(fontsize=9, markerscale=2, framealpha=0.7)

    if not any_data:
        print(f"   SKIPPED (no data): {filename}")
        plt.close()
        return

    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")


# ==========================================
# FUNCIÓN: Valor absoluto vs Ocupación
# ==========================================
def scatter_absolute(y_var, y_label, suptitle, filename):
    """Genera un scatter del valor absoluto de una variable frente a
    la ocupación, útil para evaluar rangos reales de confort."""
    month_pos = {m: i for i, m in enumerate(unique_months)}
    n_months  = len(unique_months)

    if n_months == 1:
        fig, axes = plt.subplots(1, 1, figsize=(7, 5))
        axes_flat = [axes]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes_flat = axes.flatten()

    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)

    any_data = False

    for month, pos in month_pos.items():
        ax = axes_flat[pos]
        ax.set_title(month, fontsize=12, fontweight='bold')
        ax.set_xlabel('Ocupación (personas)', fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)

        handles = []

        for sensor in SENSORS:
            sub = df_all[
                (df_all['month'] == month) &
                (df_all['sensor'] == sensor) &
                (df_all['variable'] == y_var)
            ].dropna(subset=['occupancy', 'value'])

            if len(sub) == 0:
                continue

            any_data = True
            color = sensor_colors.get(sensor, '#999999')

            sc = ax.scatter(sub['occupancy'], sub['value'],
                            c=color, alpha=0.45, s=10, edgecolors='none',
                            label=sensor)
            handles.append(sc)

            # Línea de tendencia lineal
            try:
                x_v = sub['occupancy'].values
                y_v = sub['value'].values
                if len(x_v) > 1:
                    z   = np.polyfit(x_v, y_v, 1)
                    p   = np.poly1d(z)
                    x_t = np.linspace(x_v.min(), x_v.max(), 100)
                    ax.plot(x_t, p(x_t), color=color, linewidth=1.6,
                            linestyle='--', alpha=0.9)
            except Exception:
                pass

        if handles:
            ax.legend(fontsize=9, markerscale=2, framealpha=0.7)

    if not any_data:
        print(f"   SKIPPED (no data): {filename}")
        plt.close()
        return

    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")


# ==========================================
# FUNCIÓN: CO2 vs Humedad
# ==========================================
def scatter_co2_vs_humidity(filename):
    """Genera un scatter de CO2 frente a humedad relativa, coloreando
    los puntos según el nivel de ocupación para ver si ambas variables
    crecen conjuntamente con la presencia de personas."""
    month_pos = {m: i for i, m in enumerate(unique_months)}
    n_months  = len(unique_months)

    if n_months == 1:
        fig, axes = plt.subplots(1, 1, figsize=(7, 5))
        axes_flat = [axes]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes_flat = axes.flatten()

    fig.suptitle('CO₂ vs Humedad — Aula 1.6', fontsize=14, fontweight='bold', y=1.02)

    any_data = False

    for month, pos in month_pos.items():
        ax = axes_flat[pos]
        ax.set_title(month, fontsize=12, fontweight='bold')
        ax.set_xlabel('Humedad (%)', fontsize=10)
        ax.set_ylabel('CO₂ (ppm)', fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)

        for sensor in SENSORS:
            # Extraer CO2 y humedad del mismo sensor y mes, alineados por timestamp
            co2_sub = df_all[
                (df_all['month'] == month) &
                (df_all['sensor'] == sensor) &
                (df_all['variable'] == 'CO2')
            ][['timestamp', 'value', 'occ_level']].rename(columns={'value': 'co2'})

            hum_sub = df_all[
                (df_all['month'] == month) &
                (df_all['sensor'] == sensor) &
                (df_all['variable'] == 'Humidity')
            ][['timestamp', 'value']].rename(columns={'value': 'humidity'})

            merged = pd.merge(co2_sub, hum_sub, on='timestamp').dropna()

            if len(merged) == 0:
                continue

            any_data = True

            # Colorear por nivel de ocupación
            for label, color in zip(OCC_LABELS, OCC_COLORS):
                sub = merged[merged['occ_level'] == label]
                if len(sub) == 0:
                    continue
                ax.scatter(sub['humidity'], sub['co2'],
                           c=color, alpha=0.5, s=10, edgecolors='none',
                           label=label)

        ax.legend(fontsize=8, markerscale=2, framealpha=0.7, title='Ocupación')

    if not any_data:
        print(f"   SKIPPED (no data): {filename}")
        plt.close()
        return

    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")


# ==========================================
# FUNCIÓN: Boxplot por nivel de ocupación
# ==========================================
def boxplot_by_occ_level(y_var, y_label, suptitle, filename):
    """Genera boxplots de una variable agrupados por nivel de ocupación
    (Vacío, Baja, Media, Alta) para ver la distribución de valores
    en cada rango de presencia de personas."""
    month_pos = {m: i for i, m in enumerate(unique_months)}
    n_months  = len(unique_months)

    if n_months == 1:
        fig, axes = plt.subplots(1, 1, figsize=(7, 5))
        axes_flat = [axes]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes_flat = axes.flatten()

    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)

    any_data = False

    for month, pos in month_pos.items():
        ax = axes_flat[pos]
        ax.set_title(month, fontsize=12, fontweight='bold')
        ax.set_xlabel('Nivel de ocupación', fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--', axis='y')
        ax.tick_params(labelsize=9)

        for sensor in SENSORS:
            sub = df_all[
                (df_all['month'] == month) &
                (df_all['sensor'] == sensor) &
                (df_all['variable'] == y_var)
            ].dropna(subset=['value', 'occ_level'])

            if len(sub) == 0:
                continue

            any_data = True

            # Construir lista de arrays por nivel, respetando el orden definido
            data_by_level = [
                sub[sub['occ_level'] == label]['value'].values
                for label in OCC_LABELS
            ]
            # Excluir niveles sin datos para no mostrar cajas vacías
            labels_present = [l for l, d in zip(OCC_LABELS, data_by_level) if len(d) > 0]
            data_present   = [d for d in data_by_level if len(d) > 0]

            if not data_present:
                continue

            bp = ax.boxplot(data_present, patch_artist=True,
                            medianprops=dict(color='white', linewidth=2))

            for patch, color in zip(bp['boxes'], [OCC_COLORS[OCC_LABELS.index(l)] for l in labels_present]):
                patch.set_facecolor(color)
                patch.set_alpha(0.75)

            ax.set_xticks(range(1, len(labels_present) + 1))
            ax.set_xticklabels(labels_present, fontsize=8, rotation=15, ha='right')

    if not any_data:
        print(f"   SKIPPED (no data): {filename}")
        plt.close()
        return

    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")


# ==========================================
# GENERAR GRÁFICOS — DELTAS vs OCUPACIÓN
# ==========================================
print("\n" + "="*60)
print("GENERATING DELTA SCATTER PLOTS")
print("="*60)

scatter_delta('CO2',         'ΔCO₂ (ppm)',        'ΔCO₂ vs Ocupación — Aula 1.6',         '01_dCO2_vs_Ocupacion.png')
scatter_delta('Temperature', 'ΔTemperatura (°C)', 'ΔTemperatura vs Ocupación — Aula 1.6', '02_dTemperatura_vs_Ocupacion.png')
scatter_delta('Humidity',    'ΔHumedad (%)',       'ΔHumedad vs Ocupación — Aula 1.6',     '03_dHumedad_vs_Ocupacion.png')

# ==========================================
# GENERAR GRÁFICOS — VALORES ABSOLUTOS vs OCUPACIÓN
# ==========================================
print("\n" + "="*60)
print("GENERATING ABSOLUTE VALUE SCATTER PLOTS")
print("="*60)

scatter_absolute('CO2',         'CO₂ (ppm)',        'CO₂ vs Ocupación — Aula 1.6',         '04_CO2_vs_Ocupacion.png')
scatter_absolute('Temperature', 'Temperatura (°C)', 'Temperatura vs Ocupación — Aula 1.6', '05_Temperatura_vs_Ocupacion.png')
scatter_absolute('Humidity',    'Humedad (%)',       'Humedad vs Ocupación — Aula 1.6',     '06_Humedad_vs_Ocupacion.png')

# ==========================================
# GENERAR GRÁFICO — CO2 vs HUMEDAD
# ==========================================
print("\n" + "="*60)
print("GENERATING CO2 vs HUMIDITY SCATTER")
print("="*60)

scatter_co2_vs_humidity('07_CO2_vs_Humedad.png')

# ==========================================
# GENERAR BOXPLOTS POR NIVEL DE OCUPACIÓN
# ==========================================
print("\n" + "="*60)
print("GENERATING BOXPLOTS BY OCCUPANCY LEVEL")
print("="*60)

boxplot_by_occ_level('CO2',         'CO₂ (ppm)',        'CO₂ por nivel de ocupación — Aula 1.6',         '08_Boxplot_CO2_Ocupacion.png')
boxplot_by_occ_level('Temperature', 'Temperatura (°C)', 'Temperatura por nivel de ocupación — Aula 1.6', '09_Boxplot_Temperatura_Ocupacion.png')
boxplot_by_occ_level('Humidity',    'Humedad (%)',       'Humedad por nivel de ocupación — Aula 1.6',     '10_Boxplot_Humedad_Ocupacion.png')

# ==========================================
# ANÁLISIS DE CORRELACIONES
# ==========================================
print("\n" + "="*60)
print("CORRELATION ANALYSIS (DELTAS)")
print("="*60)

def correlation_by_occupation(variable, min_occupancy=0):
    """Calcula la correlación de Pearson entre ocupación y delta de una variable,
    filtrando por un mínimo de ocupación para aislar periodos relevantes."""
    for sensor in SENSORS:
        sub = df_all[
            (df_all['sensor'] == sensor) &
            (df_all['variable'] == variable) &
            (df_all['occupancy'] >= min_occupancy)
        ].dropna(subset=['occupancy', 'delta_value'])

        if len(sub) > 5:
            corr = sub['occupancy'].corr(sub['delta_value'])
            print(f"   {sensor} - {variable}: r = {corr:.3f} (n={len(sub)})")

print("\nCorrelaciones (ocupación ≥ 0):")
for var in VARIABLES:
    correlation_by_occupation(var)

print("\nCorrelaciones (ocupación ≥ 3):")
for var in VARIABLES:
    correlation_by_occupation(var, min_occupancy=3)

print("\n" + "="*60)
print("ANÁLISIS COMPLETADO — AULA 1.6")
print("="*60)


Found 12 weekly files

LOADING ALL DATA...
   1.5_data_December_2025_week_4_days_22-31.xlsx... OK
   1.5_data_February_2026_week_1_days_1-7.xlsx... OK
   1.5_data_February_2026_week_2_days_8-14.xlsx... OK
   1.5_data_February_2026_week_3_days_15-21.xlsx... OK
   1.5_data_February_2026_week_4_days_22-28.xlsx... OK
   1.5_data_January_2026_week_1_days_1-7.xlsx... OK
   1.5_data_January_2026_week_2_days_8-14.xlsx... OK
   1.5_data_January_2026_week_3_days_15-21.xlsx... OK
   1.5_data_January_2026_week_4_days_22-31.xlsx... OK
   1.5_data_March_2026_week_1_days_1-7.xlsx... OK
   1.5_data_March_2026_week_2_days_8-14.xlsx... OK
   1.5_data_March_2026_week_3_days_15-21.xlsx... OK

   Total records: 69837

Calculating deltas...
   Records after filter (Feb+Mar): 60294

GENERATING DELTA SCATTER PLOTS
   SKIPPED (no data): 17_dCO2_vs_Occupancy.png
   SKIPPED (no data): 18_dTemperature_vs_Occupancy.png
   SKIPPED (no data): 19_dHumidity_vs_Occupancy.png

CORRELATION ANALYSIS (DELTAS)

Correlation